In [0]:
%pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.4/795.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 167.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install databricks-vectorsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-35f3904c-952e-4ea8-ba4a-81260ab065b7
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-35f3904c-952e-4ea8-ba4a-81260ab065b7
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-35f3904c-952e-4ea8-

In [0]:
dbutils.library.restartPython()

In [0]:
# Vector Search client
from databricks.vector_search.client import VectorSearchClient

vs_client = VectorSearchClient()

# HTTP for Sarvam API calls
import requests

# (optional safety check)
print("✅ Backend initialized: Vector Search + HTTP ready")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
✅ Backend initialized: Vector Search + HTTP ready


In [0]:
vs_client.get_index(
    endpoint_name="export_ai_vector_endpoint",
    index_name="export_ai.rag.document_chunks_index"
).describe()

{'name': 'export_ai.rag.document_chunks_index',
 'endpoint_name': 'export_ai_vector_endpoint',
 'primary_key': 'id',
 'index_type': 'DELTA_SYNC',
 'delta_sync_index_spec': {'source_table': 'export_ai.rag.document_chunks',
  'embedding_source_columns': [{'name': 'content',
    'embedding_model_endpoint_name': 'databricks-gte-large-en'}],
  'pipeline_type': 'TRIGGERED',
  'pipeline_id': '44c53fdd-4ec0-4c5a-8ef5-2d81b245d43d'},
 'status': {'detailed_state': 'ONLINE_NO_PENDING_UPDATE',
  'message': 'Index creation succeeded. Check latest status: https://dbc-21bb95c1-0f91.cloud.databricks.com/explore/data/export_ai/rag/document_chunks_index',
  'indexed_row_count': 811,
  'triggered_update_status': {'last_processed_commit_version': 1,
   'last_processed_commit_timestamp': '2026-04-18T07:54:22Z'},
  'ready': True,
  'index_url': 'dbc-21bb95c1-0f91.cloud.databricks.com/api/2.0/vector-search/indexes/export_ai.rag.document_chunks_index'},
 'creator': 'che240008016@iiti.ac.in',
 'endpoint_type':

In [0]:
sarvam_api_key = dbutils.secrets.get(
    scope="api_keys_scope",
    key="sarvam_api_key"
)

In [0]:
# ===============================
# MAIN BACKEND RAG FUNCTION
# ===============================

def export_ai_pipeline(query):

    # 1. Retrieve relevant chunks from Vector Search

    index = vs_client.get_index(
    endpoint_name="export_ai_vector_endpoint",  # your endpoint name
    index_name="export_ai.rag.document_chunks_index"
    )

    results = index.similarity_search(
    query_text=query,
    columns=["content"],
    num_results=5
    )

    rows = results["result"]["data_array"]

    context = "\n\n".join([r[0] for r in rows])
    # 3. Call Sarvam AI (reasoning + response generation)
    payload = {
        "input": f"""
You are an export intelligence assistant for India.

Use the context below to answer the question.

CONTEXT:
{context}

QUESTION:
{query}

Return structured output with:
- Summary
- Export opportunities
- Risks
- Recommended countries
""",
        "source_language_code": "en-IN",
        "target_language_code": "en-IN"
    }

    response = requests.post(
    "https://api.sarvam.ai/v1/chat/completions",
    json={
        "model": "sarvam-m",
        "messages": [
            {
                "role": "user",
                "content": f"""
You are an export intelligence assistant.

Use the context below:

{context}

Question:
{query}

Give structured export insights.
"""
            }
        ]
    },
    headers={
        "api-subscription-key": sarvam_api_key
    }
)

    # 4. Return final answer
    return response.json()["choices"][0]["message"]["content"]

    import re
    cleaned = re.sub(r"<think>.*?</think>", "", data, flags=re.DOTALL)

    return cleaned.strip()

In [0]:
export_ai_pipeline("rice export opportunities from India to UAE")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


"<think>\nOkay, let's tackle this query about rice export opportunities from India to the UAE. First, I need to understand what the user is asking for. They want structured export insights, so I should organize the information clearly.\n\nLooking at the context provided, there's data on India's rice exports to Canada from 2018 to 2021. But the user is asking about the UAE, which isn't directly covered in the data. However, maybe there's some transferable information or legal frameworks that apply to exports in general, not just Canada.\n\nThe context mentions several legal frameworks and trade facilitation measures. For example, the single window system through APEDA for perishable agricultural products, the Niryat Bandhu Scheme for mentoring exporters, and the DGFT portal. These could be relevant for UAE exports too.\n\nAlso, the Staple Food Control Act and Salt Industry Act are mentioned, but those might be specific to domestic regulations. However, export-related regulations might b

2026-04-18 10:42:35.797 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 10:42:35.804 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 10:42:35.805 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 10:42:35.805 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 10:42:35.807 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 10:42:35.807 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 10:42:35.808 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 10:42:35.809 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar